# Adjudication Error-Bar Analysis

This notebook contains two ordered data pipelines:

1. **RA analysis** loads and analyzes `adjudicated_ra_affinities.csv`.
2. **PRP analysis** runs only when `adjudicated_prp_affinities.csv` is present.

The PRP section also compares direct PRP counts with RA-to-PRP counts when both inputs are available.

In [64]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px

In [65]:
# Shared run selection
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.analysis.prp_assignment import (
    apply_affinity_level_fixed,
    assign_prps,
)

bench_root = repo_root / 'data' / 'results' / 'affinity_benchmark'
run_dirs = sorted(path for path in bench_root.iterdir() if path.is_dir())
if not run_dirs:
    raise FileNotFoundError(f'No benchmark runs found under {bench_root}')
run_dir = run_dirs[-1]

print('Run directory:', run_dir)

ImportError: cannot import name 'assign_prps' from 'src.analysis.prp_assignment' (/mnt/c/Users/pgallego/OneDrive - Imperial College London/PhD/Repositories/article-classifier/src/analysis/prp_assignment.py)

# 1. RA Analysis

Map adjudicated RA questions to programmes, apply fixed affinity levels, and assign each abstract to one or more programmes from its strongest question scores.

In [ ]:
# RA input and programme assignment
ra_path_adjudication = run_dir / 'adjudicated_ra_affinities.csv'
if not ra_path_adjudication.exists():
    raise FileNotFoundError(f'RA adjudication input not found: {ra_path_adjudication}')

ra_adjudication_df = pd.read_csv(ra_path_adjudication)

# Low programmes are discarded and High programmes are always retained.
# The Moderate-programme margin is configured in settings.yaml.
ra_analysis_df = assign_prps(ra_adjudication_df)

print('RA input rows:', len(ra_adjudication_df))
print('Selected abstract-PRP assignments:', len(ra_analysis_df))
display(ra_analysis_df.head(10))

RA input rows: 17184
Mapped RA question rows: 17184
Selected abstract-PRP assignments: 492


,Abstract_Index,PRP_Name,Highest_Question_ID,Max_Affinity,Highest_Question_Text,Best_Abstract_Affinity,Affinity_Gap,Selection_Reason,Affinity_Level
0,10.1109/TPWRS.2025.3560178,CROF,24,77.6,how can the risk of high-impact low-probabilit...,77.6,0.0,within_relative_margin,Moderate
1,10.1109/TPWRS.2025.3560178,Planning,46,74.6,"what additional methods, tools, and metrics ar...",77.6,3.0,within_relative_margin,Moderate
2,10.1109/TPWRS.2025.3585527,Stability,8,83.2,"what methods, including data types, and analys...",83.2,0.0,high_affinity,High
3,10.1109/TPWRS.2025.3586137,System Services,33,93.6,how should the definitions of services for ibr...,93.6,0.0,high_affinity,High
4,10.1109/TPWRS.2025.3586137,DER,18,84.6,what are the relative merits of different coor...,93.6,9.0,high_affinity,High
5,10.1109/TPWRS.2025.3586137,IBR,1,82.0,what is the future of active power balance as ...,93.6,11.6,high_affinity,High
6,10.1109/TPWRS.2025.3586746,CROF,24,85.4,how can the risk of high-impact low-probabilit...,85.4,0.0,high_affinity,High
7,10.1109/TPWRS.2025.3586788,DER,16,78.4,what is the communication capability needed to...,78.4,0.0,within_relative_margin,Moderate
8,10.1109/TPWRS.2025.3586790,CROF,26,41.6,what improvements can be made to predictive to...,41.6,0.0,within_relative_margin,Moderate
9,10.1109/TPWRS.2025.3586960,Planning,39,69.4,what technical models need to be added to long...,69.4,0.0,within_relative_margin,Moderate


In [ ]:
# Deterministic counts from selected RA-to-PRP assignments
ra_count_df = (
    ra_analysis_df.groupby(
        ['PRP_Name', 'Affinity_Level'],
        as_index=False,
        observed=True,
    )['Abstract_Index']
    .nunique()
    .rename(columns={'Abstract_Index': 'n_abstracts'})
)

fig = px.bar(
    ra_count_df,
    x='PRP_Name',
    y='n_abstracts',
    color='Affinity_Level',
    barmode='group',
    category_orders={'Affinity_Level': ['Moderate', 'High']},
    title='Selected RA-to-PRP Assignments by Affinity Level and PRP',
)
fig.update_layout(
    xaxis_title='PRP Name',
    yaxis_title='Number of Abstracts',
    legend_title='Affinity Level',
)
fig.update_xaxes(tickangle=45)
fig.show()

display(ra_count_df.sort_values(['PRP_Name', 'Affinity_Level']))

,PRP_Name,Affinity_Level,n_abstracts
0,CROF,Moderate,98
1,CROF,High,27
2,DER,Moderate,24
3,DER,High,13
4,IBR,Moderate,35
5,IBR,High,20
6,Planning,Moderate,61
7,Planning,High,17
8,Stability,Moderate,82
9,Stability,High,56


# 2. PRP Analysis

This section loads, processes, and plots direct PRP adjudication results only when `adjudicated_prp_affinities.csv` exists.

In [ ]:
# PRP input and processing
prp_path_adjudication = run_dir / 'adjudicated_prp_affinities.csv'
prp_available = prp_path_adjudication.exists()

if prp_available:
    prp_adjudication_df = pd.read_csv(prp_path_adjudication)
    prp_analysis_df = prp_adjudication_df.copy()
    prp_analysis_df['Final_Affinity'] = pd.to_numeric(
        prp_analysis_df['Final_Affinity'],
        errors='coerce',
    )
    prp_analysis_df['Affinity_Mean'] = pd.to_numeric(
        prp_analysis_df['Affinity_Mean'],
        errors='coerce',
    )
    prp_analysis_df['Affinity_Std'] = pd.to_numeric(
        prp_analysis_df['Affinity_Std'],
        errors='coerce',
    ).fillna(0.0)
    prp_analysis_df = prp_analysis_df.dropna(
        subset=['Final_Affinity', 'PRP_Name', 'Abstract_Index'],
    ).copy()
    prp_analysis_df['err_low'] = (
        prp_analysis_df['Final_Affinity'] - prp_analysis_df['Affinity_Std']
    ).clip(lower=0)
    prp_analysis_df['err_high'] = (
        prp_analysis_df['Final_Affinity'] + prp_analysis_df['Affinity_Std']
    ).clip(upper=120)
    prp_analysis_df = apply_affinity_level_fixed(prp_analysis_df)

    print('PRP input rows:', len(prp_adjudication_df))
    print('Prepared PRP rows:', len(prp_analysis_df))
    display(prp_analysis_df.head(10))
else:
    print(f'PRP analysis skipped: input file not found at {prp_path_adjudication}')

PRP analysis skipped: input file not found at /mnt/c/Users/pgallego/OneDrive - Imperial College London/PhD/Repositories/article-classifier/data/results/affinity_benchmark/tpwrs_2026_tpwrs_2026/adjudicated_prp_affinities.csv


## PRP Affinity and Error Bars

In [ ]:
if prp_available:
    plot_df = prp_analysis_df.sort_values(['PRP_Name', 'Abstract_Index']).copy()

    fig = px.box(
        plot_df,
        x='PRP_Name',
        y='Final_Affinity',
        color='Affinity_Level',
        points='all',
        hover_data=['Abstract_Index', 'Affinity_Mean', 'Affinity_Std', 'err_low', 'err_high'],
        category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
        title='Final Affinity Distribution by PRP and Affinity Level',
    )
    fig.update_layout(
        xaxis_title='PRP Name',
        yaxis_title='Final Affinity',
        legend_title='Affinity Level',
    )
    fig.update_xaxes(tickangle=45)
    fig.show()

    top_uncertain = (
        plot_df.nlargest(40, 'Affinity_Std')
        .assign(Pair_Label=lambda frame: frame['Abstract_Index'].astype(str) + ' | ' + frame['PRP_Name'])
        .sort_values('Affinity_Std', ascending=True)
    )
    fig = px.bar(
        top_uncertain,
        x='Affinity_Std',
        y='Pair_Label',
        color='Affinity_Level',
        orientation='h',
        hover_data=['Final_Affinity', 'Affinity_Mean', 'err_low', 'err_high'],
        category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
        title='Top 40 Most Uncertain (Abstract, PRP) Pairs',
    )
    fig.update_layout(
        xaxis_title='Affinity Standard Deviation',
        yaxis_title='Abstract | PRP',
        height=900,
    )
    fig.show()

    fig = px.scatter(
        plot_df,
        x='Abstract_Index',
        y='Final_Affinity',
        color='Affinity_Level',
        error_y='Affinity_Std',
        facet_col='PRP_Name',
        facet_col_wrap=3,
        hover_data=['Affinity_Mean', 'Affinity_Std', 'err_low', 'err_high'],
        category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
        title='Final Affinity with Error Bars by Abstract (Faceted by PRP)',
    )
    fig.update_layout(yaxis_title='Final Affinity', height=900)
    fig.show()
else:
    print('PRP affinity and error-bar plots skipped.')

PRP affinity and error-bar plots skipped.


## PRP Summaries and Counts

In [ ]:
if prp_available:
    prp_summary = (
        prp_analysis_df.groupby('PRP_Name', as_index=False)
        .agg(
            n=('Final_Affinity', 'count'),
            mean_final_affinity=('Final_Affinity', 'mean'),
            mean_row_std=('Affinity_Std', 'mean'),
        )
    )
    prp_mean_err = (
        prp_analysis_df.assign(variance=lambda frame: frame['Affinity_Std'] ** 2)
        .groupby('PRP_Name', as_index=False)['variance']
        .sum()
        .merge(prp_summary[['PRP_Name', 'n']], on='PRP_Name', how='left')
    )
    prp_mean_err['mean_error'] = np.sqrt(prp_mean_err['variance']) / prp_mean_err['n']
    prp_summary = prp_summary.merge(
        prp_mean_err[['PRP_Name', 'mean_error']],
        on='PRP_Name',
        how='left',
    )

    fig = px.bar(
        prp_summary.sort_values('mean_final_affinity', ascending=False),
        x='PRP_Name',
        y='mean_final_affinity',
        error_y='mean_error',
        title='PRP Mean Final Affinity with Propagated Error Bars',
    )
    fig.update_layout(xaxis_title='PRP Name', yaxis_title='Mean Final Affinity')
    fig.update_xaxes(tickangle=45)
    fig.show()
    display(prp_summary.sort_values('mean_final_affinity', ascending=False))

    prp_count_df = (
        prp_analysis_df.groupby(['PRP_Name', 'Affinity_Level'], as_index=False)['Abstract_Index']
        .nunique()
        .rename(columns={'Abstract_Index': 'n_abstracts'})
    )
    fig = px.bar(
        prp_count_df,
        x='PRP_Name',
        y='n_abstracts',
        color='Affinity_Level',
        barmode='group',
        category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
        title='Direct PRP Abstract Counts by Affinity Level and PRP',
    )
    fig.update_layout(
        xaxis_title='PRP Name',
        yaxis_title='Number of Abstracts',
        legend_title='Affinity Level',
    )
    fig.update_xaxes(tickangle=45)
    fig.show()
    display(prp_count_df.sort_values(['PRP_Name', 'Affinity_Level']))
else:
    print('PRP summaries and deterministic counts skipped.')

PRP summaries and deterministic counts skipped.


In [ ]:
if prp_available:
    n_simulations = 2000
    rng = np.random.default_rng(42)
    simulation_base = prp_analysis_df[
        ['PRP_Name', 'Final_Affinity', 'Affinity_Std']
    ].copy()
    simulation_base['Affinity_Std'] = simulation_base['Affinity_Std'].clip(lower=0.0)

    simulated_counts = []
    for simulation_index in range(n_simulations):
        sampled_affinity = rng.normal(
            loc=simulation_base['Final_Affinity'].to_numpy(),
            scale=simulation_base['Affinity_Std'].to_numpy(),
        )
        sampled_affinity = np.clip(sampled_affinity, 0, 120)
        sampled_level = np.where(
            sampled_affinity <= 40,
            'Low',
            np.where(sampled_affinity <= 80, 'Moderate', 'High'),
        )
        simulation_df = pd.DataFrame(
            {
                'PRP_Name': simulation_base['PRP_Name'].to_numpy(),
                'Affinity_Level': sampled_level,
            }
        )
        simulation_count = simulation_df.value_counts(
            ['PRP_Name', 'Affinity_Level']
).reset_index(name='n_abstracts')
        simulation_count['simulation'] = simulation_index
        simulated_counts.append(simulation_count)

    simulated_count_df = pd.concat(simulated_counts, ignore_index=True)
    count_uncertainty = (
        simulated_count_df.groupby(['PRP_Name', 'Affinity_Level'], as_index=False)['n_abstracts']
        .agg(mean_count='mean', std_count='std')
        .merge(prp_count_df, on=['PRP_Name', 'Affinity_Level'], how='left')
    )

    fig = px.bar(
        count_uncertainty,
        x='PRP_Name',
        y='mean_count',
        error_y='std_count',
        color='Affinity_Level',
        barmode='group',
        category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
        title='PRP Abstract Counts with Propagated Error Bars',
    )
    fig.update_layout(
        xaxis_title='PRP Name',
        yaxis_title='Expected Number of Abstracts',
        legend_title='Affinity Level',
    )
    fig.update_xaxes(tickangle=45)
    fig.show()
    display(count_uncertainty.sort_values(['PRP_Name', 'Affinity_Level']))
else:
    print('PRP Monte Carlo count uncertainty skipped.')

PRP Monte Carlo count uncertainty skipped.


## RA-to-PRP versus Direct PRP Counts

In [ ]:
if prp_available:
    combined_count_df = pd.concat(
        [
            prp_count_df.assign(source='Direct PRP'),
            ra_count_df.assign(source='RA-to-PRP'),
        ],
        ignore_index=True,
    )
    display(
        combined_count_df.sort_values(
            ['source', 'PRP_Name', 'Affinity_Level']
)
    )

    fig = px.bar(
        combined_count_df[combined_count_df['Affinity_Level'] != 'Low'],
        x='PRP_Name',
        y='n_abstracts',
        facet_col='Affinity_Level',
        color='source',
        barmode='group',
        category_orders={
            'Affinity_Level': ['Moderate', 'High'],
            'source': ['Direct PRP', 'RA-to-PRP'],
        },
        title='Abstract Counts by Affinity Level: Direct PRP versus RA-to-PRP',
    )
    fig.update_layout(
        xaxis_title='PRP Name',
        yaxis_title='Number of Abstracts',
        legend_title='Source',
    )
    fig.update_xaxes(tickangle=45)
    fig.show()
else:
    print('RA-to-PRP versus direct PRP comparison skipped.')

RA-to-PRP versus direct PRP comparison skipped.
